## Problem 1: Reversing a singly linked list

In [1]:
import time


class Node:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next


def build_list(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head


def to_string(head):
    out = []
    while head is not None:
        out.append(str(head.val))
        head = head.next
    out.append("None")
    return " -> ".join(out)


def reverse(head):
    prev, curr = None, head
    while curr is not None:
        nxt = curr.next      # save the rest of the list
        curr.next = prev     # flip the pointer
        prev, curr = curr, nxt
    return prev              # old tail is the new head

In [2]:
head = build_list([1, 2, 3, 4, 5, 6])
print("original:", to_string(head))
print("reversed:", to_string(reverse(head)))
print("empty   :", to_string(reverse(None)))
print("single  :", to_string(reverse(build_list([7]))))

original: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
reversed: 6 -> 5 -> 4 -> 3 -> 2 -> 1 -> None
empty   : None
single  : 7 -> None


In [3]:
# time per node should stay constant if the reversal is O(n)
print(f"{'n':>8} {'time (ms)':>10} {'ns/node':>8}")
for n in [200_000, 400_000, 800_000, 1_600_000]:
    h = build_list(list(range(n)))
    t0 = time.perf_counter()
    reverse(h)
    t = time.perf_counter() - t0
    print(f"{n:>8} {t*1e3:>10.2f} {t/n*1e9:>8.1f}")

       n  time (ms)  ns/node
  200000      16.51     82.6


  400000      33.48     83.7


  800000      67.21     84.0


 1600000     138.35     86.5


## Problem 2: Tower of Hanoi with three stacks

In [4]:
class Stack:
    def __init__(self, name):
        self.name = name
        self.items = []

    def push(self, disk):
        if self.items and self.items[-1] < disk:
            raise ValueError(f"cannot put disk {disk} on {self.items[-1]}")
        self.items.append(disk)

    def pop(self):
        return self.items.pop()

    def __repr__(self):
        return f"{self.name}: {self.items}"


def hanoi(n, src, dst, aux, moves):
    if n == 0:
        return
    hanoi(n - 1, src, aux, dst, moves)    # top n-1 disks out of the way
    disk = src.pop()                      # largest disk
    dst.push(disk)
    moves.append((disk, src.name, dst.name))
    hanoi(n - 1, aux, dst, src, moves)    # n-1 disks back on top


def solve(n, verbose=False):
    A, B, C = Stack("A"), Stack("B"), Stack("C")
    for d in range(n, 0, -1):
        A.push(d)
    moves = []
    if verbose:
        print("start:", A, B, C)
    hanoi(n, A, C, B, moves)
    if verbose:
        for i, (d, s, t) in enumerate(moves, 1):
            print(f"move {i}: disk {d} {s} -> {t}")
        print("end  :", A, B, C)
    return len(moves)

In [5]:
solve(3, verbose=True);

start: A: [3, 2, 1] B: [] C: []
move 1: disk 1 A -> C
move 2: disk 2 A -> B
move 3: disk 1 C -> B
move 4: disk 3 A -> C
move 5: disk 1 B -> A
move 6: disk 2 B -> C
move 7: disk 1 A -> C
end  : A: [] B: [] C: [3, 2, 1]


In [6]:
print(f"{'n':>2} {'moves':>6} {'2^n-1':>6}")
for n in range(1, 11):
    print(f"{n:>2} {solve(n):>6} {2**n - 1:>6}")

 n  moves  2^n-1
 1      1      1
 2      3      3
 3      7      7
 4     15     15
 5     31     31
 6     63     63
 7    127    127
 8    255    255
 9    511    511
10   1023   1023


## Problem 3: Rotations and balancing a BST

In [7]:
class TNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None


def insert(root, key):
    if root is None:
        return TNode(key)
    if key < root.key:
        root.left = insert(root.left, key)
    else:
        root.right = insert(root.right, key)
    return root


def height(root):
    return 0 if root is None else 1 + max(height(root.left), height(root.right))


def size(root):
    return 0 if root is None else 1 + size(root.left) + size(root.right)


def show(root):
    if root is None:
        return "-"
    if root.left is None and root.right is None:
        return str(root.key)
    return f"{root.key}({show(root.left)},{show(root.right)})"

In [8]:
rotations = []


def rotate_left(x):
    y = x.right
    x.right = y.left
    y.left = x
    rotations.append(("left", x.key))
    return y


def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    rotations.append(("right", y.key))
    return x

### (a), (b) Small examples

In [9]:
def build(keys):
    r = None
    for k in keys:
        r = insert(r, k)
    return r

r = build([30, 20, 10])
print("left-left  :", show(r), "-> right at 30 ->", show(rotate_right(r)))

r = build([10, 20, 30])
print("right-right:", show(r), "-> left at 10 ->", show(rotate_left(r)))

r = build([30, 10, 20])
print("left-right :", show(r), end=" ")
r.left = rotate_left(r.left)
print("-> left at 10 ->", show(r), end=" ")
print("-> right at 30 ->", show(rotate_right(r)))

left-left  : 30(20(10,-),-) -> right at 30 -> 20(10,30)
right-right: 10(-,20(-,30)) -> left at 10 -> 20(10,30)
left-right : 30(10(-,20),-) -> left at 10 -> 30(20(10,-),-) -> right at 30 -> 20(10,30)


### (c) Insert 10, 20, ..., 100 and balance with rotations

In [10]:
def kth(root, k):
    # key with exactly k smaller keys in this subtree
    left = size(root.left)
    if k < left:
        return kth(root.left, k)
    if k == left:
        return root.key
    return kth(root.right, k - left - 1)


def bring_to_root(root, key):
    if key < root.key:
        root.left = bring_to_root(root.left, key)
        return rotate_right(root)
    if key > root.key:
        root.right = bring_to_root(root.right, key)
        return rotate_left(root)
    return root


def balance(root):
    # make the median the root of every subtree, using rotations only
    if root is None:
        return None
    median = kth(root, (size(root) - 1) // 2)
    before = len(rotations)
    root = bring_to_root(root, median)
    if len(rotations) > before:
        print(f"median {median}: {rotations[before:]}")
        print(f"    -> {show(root)}")
    root.left = balance(root.left)
    root.right = balance(root.right)
    return root

In [11]:
rotations.clear()
root = build(range(10, 101, 10))
print("after insertion:", show(root), "| height", height(root))
root = balance(root)
print("balanced:", show(root), "| height", height(root), "| rotations", len(rotations))

after insertion: 10(-,20(-,30(-,40(-,50(-,60(-,70(-,80(-,90(-,100))))))))) | height 10
median 50: [('left', 40), ('left', 30), ('left', 20), ('left', 10)]
    -> 50(10(-,20(-,30(-,40))),60(-,70(-,80(-,90(-,100)))))
median 20: [('left', 10)]
    -> 20(10,30(-,40))
median 80: [('left', 70), ('left', 60)]
    -> 80(60(-,70),90(-,100))
balanced: 50(20(10,30(-,40)),80(60(-,70),90(-,100))) | height 4 | rotations 7


## Problem 4: Entropy of binary trees

In [12]:
import math
import random
import itertools
from functools import lru_cache

random.seed(38)
values = random.sample(range(10, 100), 15)    # 15 distinct numbers in [10, 99]
print(values)


def build_bst(order):
    root = None
    for k in order:
        root = insert(root, k)
    return root


def fill_weights(v):
    if v is None:
        return 0
    v.W = v.key + fill_weights(v.left) + fill_weights(v.right)
    return v.W


def entropy(root):
    """Sets v.H for every node and returns the total entropy S."""
    fill_weights(root)
    S, stack = 0.0, [root]
    while stack:
        v = stack.pop()
        v.H = 0.0
        for c in (v.left, v.right):
            if c is not None:
                p = c.W / v.W
                v.H -= p * math.log2(p)
                stack.append(c)
        S += v.H
    return S


def inorder_nodes(v):
    return [] if v is None else inorder_nodes(v.left) + [v] + inorder_nodes(v.right)


def preorder_keys(v):
    return [] if v is None else [v.key] + preorder_keys(v.left) + preorder_keys(v.right)

[91, 63, 64, 23, 18, 56, 99, 69, 57, 15, 87, 95, 31, 85, 54]


### (a) Tree in drawn order

In [13]:
root = build_bst(values)
S = entropy(root)
print("tree  :", show(root), "| height", height(root))
print(f"{'node':>5} {'W_v':>5} {'H(v)':>8}")
for v in inorder_nodes(root):
    print(f"{v.key:>5} {v.W:>5} {v.H:>8.4f}")
print(f"S = {S:.4f}")

tree  : 91(63(23(18(15,-),56(31(-,54),57)),64(-,69(-,87(85,-)))),99(95,-)) | height 6
 node   W_v     H(v)
   15    15   0.0000
   18    33   0.5170
   23   254   0.6626
   31    85   0.4158
   54    54   0.0000
   56   198   1.0409
   57    57   0.0000
   63   622   1.0318
   64   305   0.2685
   69   241   0.3473
   85    85   0.0000
   87   172   0.5025
   91   907   0.8491
   95    95   0.0000
   99   194   0.5044
S = 6.1400


### (b) Entropy against shape

In [14]:
for name, order in [("ascending", sorted(values)), ("descending", sorted(values, reverse=True))]:
    t = build_bst(order)
    print(f"{name:<10} chain: height {height(t)}, S = {entropy(t):.4f}")

trials = 100_000
by_height = {}
best, worst = (-1.0, None), (1e9, None)
for _ in range(trials):
    order = values[:]
    random.shuffle(order)
    t = build_bst(order)
    s = entropy(t)
    by_height.setdefault(height(t), []).append(s)
    best = max(best, (s, order))
    worst = min(worst, (s, order))

print(f"\n{'height':>6} {'trees':>6} {'mean S':>7} {'min S':>7} {'max S':>7}")
for h in sorted(by_height):
    a = by_height[h]
    print(f"{h:>6} {len(a):>6} {sum(a)/len(a):>7.4f} {min(a):>7.4f} {max(a):>7.4f}")

ascending  chain: height 15, S = 2.3112
descending chain: height 15, S = 3.9920



height  trees  mean S   min S   max S
     4      2  7.0313  7.0313  7.0313
     5   7009  6.5225  5.6614  7.2734
     6  34622  6.1339  4.9226  7.2459
     7  34564  5.7425  4.5643  7.0999
     8  16944  5.3510  4.1103  6.7733
     9   5460  4.9394  3.8045  6.4771
    10   1201  4.5120  3.4031  5.9859
    11    175  4.1393  3.1361  5.3328
    12     23  3.7040  3.0636  4.4986


### (c) Maximum and minimum entropy

In [15]:
print(f"shuffling ({trials} orders): max S = {best[0]:.4f}, min S = {worst[0]:.4f}")

shuffling (100000 orders): max S = 7.2734, min S = 3.0636


In [16]:
def plogp(part, whole):
    if part == 0:
        return 0.0
    p = part / whole
    return -p * math.log2(p)


def optimal_tree(nums, pick):
    """Exact best S over all BSTs on nums (pick = max or min), by interval DP."""
    keys = sorted(nums)
    pre = [0]
    for k in keys:
        pre.append(pre[-1] + k)
    w = lambda i, j: pre[j + 1] - pre[i] if i <= j else 0

    @lru_cache(maxsize=None)
    def dp(i, j):
        if i > j:
            return (0.0, None)
        W = w(i, j)
        return pick((plogp(w(i, r - 1), W) + plogp(w(r + 1, j), W)
                     + dp(i, r - 1)[0] + dp(r + 1, j)[0], r)
                    for r in range(i, j + 1))

    def rebuild(i, j):
        if i > j:
            return None
        r = dp(i, j)[1]
        node = TNode(keys[r])
        node.left, node.right = rebuild(i, r - 1), rebuild(r + 1, j)
        return node

    return dp(0, len(keys) - 1)[0], rebuild(0, len(keys) - 1)


for name, pick in [("max", max), ("min", min)]:
    s, t = optimal_tree(values, pick)
    order = preorder_keys(t)          # inserting in pre-order rebuilds the tree
    print(f"{name}: S = {s:.4f} (rebuilt from order: {entropy(build_bst(order)):.4f}), height {height(t)}")
    print("   tree :", show(t))
    print("   order:", order)

max: S = 7.2734 (rebuilt from order: 7.2734), height 5
   tree : 69(56(31(18(15,23),54),63(57,64)),91(87(85,-),99(95,-)))
   order: [69, 56, 31, 18, 15, 23, 54, 63, 57, 64, 91, 87, 85, 99, 95]
min: S = 2.3112 (rebuilt from order: 2.3112), height 15
   tree : 15(-,18(-,23(-,31(-,54(-,56(-,57(-,63(-,64(-,69(-,85(-,87(-,91(-,95(-,99))))))))))))))
   order: [15, 18, 23, 31, 54, 56, 57, 63, 64, 69, 85, 87, 91, 95, 99]


In [17]:
# check the DP against brute force over all 7! orders of 7 numbers
small = values[:7]
brute = [entropy(build_bst(p)) for p in itertools.permutations(small)]
print(f"brute force: max {max(brute):.4f}, min {min(brute):.4f}")
print(f"DP         : max {optimal_tree(small, max)[0]:.4f}, min {optimal_tree(small, min)[0]:.4f}")

brute force: max 3.1086, min 1.4016
DP         : max 3.1086, min 1.4016


## Problem 5: Rebuilding a tree from traversals

In [18]:
import sys
from collections import Counter

sys.setrecursionlimit(20000)


class BNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def preorder(t):
    return [] if t is None else [t.val] + preorder(t.left) + preorder(t.right)


def inorder(t):
    return [] if t is None else inorder(t.left) + [t.val] + inorder(t.right)


def postorder(t):
    return [] if t is None else postorder(t.left) + postorder(t.right) + [t.val]


def show_b(t):
    if t is None:
        return "-"
    if t.left is None and t.right is None:
        return str(t.val)
    return f"{t.val}({show_b(t.left)},{show_b(t.right)})"


pre = list("ABDEHIKCFGJ")
ino = list("DBHEKIAFCJG")
post = list("DHKIEBFJGCA")

### (a), (e) Pre-order + in-order: scan version and dictionary version

In [19]:
def build_scan(pre, ino):
    """Root found by scanning the in-order window. Returns (tree, comparisons)."""
    pos, comparisons = 0, 0

    def rec(lo, hi):
        nonlocal pos, comparisons
        if lo > hi:
            return None
        root_val = pre[pos]
        pos += 1
        k = lo
        while True:
            comparisons += 1
            if ino[k] == root_val:
                break
            k += 1
        node = BNode(root_val)
        node.left = rec(lo, k - 1)
        node.right = rec(k + 1, hi)
        return node

    return rec(0, len(ino) - 1), comparisons


def build_dict(pre, ino):
    """Root found with a label -> in-order index dictionary. O(n)."""
    where = {v: i for i, v in enumerate(ino)}
    pos = 0

    def rec(lo, hi):
        nonlocal pos
        if lo > hi:
            return None
        root_val = pre[pos]
        pos += 1
        k = where[root_val]
        node = BNode(root_val)
        node.left = rec(lo, k - 1)
        node.right = rec(k + 1, hi)
        return node

    return rec(0, len(ino) - 1)


t1 = build_dict(pre, ino)
print("tree            :", show_b(t1))
print("post-order      :", ", ".join(postorder(t1)))
print("matches list (5):", postorder(t1) == post)
print("scan gives same :", show_b(build_scan(pre, ino)[0]) == show_b(t1))

tree            : A(B(D,E(H,I(K,-))),C(F,G(J,-)))
post-order      : D, H, K, I, E, B, F, J, G, C, A
matches list (5): True
scan gives same : True


### (b) In-order + post-order

In [20]:
def build_post(post, ino):
    where = {v: i for i, v in enumerate(ino)}
    pos = len(post) - 1                 # read post-order from the back

    def rec(lo, hi):
        nonlocal pos
        if lo > hi:
            return None
        root_val = post[pos]
        pos -= 1
        k = where[root_val]
        node = BNode(root_val)
        node.right = rec(k + 1, hi)     # right subtree comes first
        node.left = rec(lo, k - 1)
        return node

    return rec(0, len(ino) - 1)


t2 = build_post(post, ino)
print("tree         :", show_b(t2))
print("pre-order    :", ", ".join(preorder(t2)))
print("same as (a)  :", show_b(t2) == show_b(t1))

tree         : A(B(D,E(H,I(K,-))),C(F,G(J,-)))
pre-order    : A, B, D, E, H, I, K, C, F, G, J
same as (a)  : True


### (c) Same pre-order and post-order, different trees

In [21]:
x = BNode(1, left=BNode(2))
y = BNode(1, right=BNode(2))
for t in (x, y):
    print(f"{show_b(t):<7} pre {preorder(t)}  post {postorder(t)}  in {inorder(t)}")

1(2,-)  pre [1, 2]  post [2, 1]  in [2, 1]
1(-,2)  pre [1, 2]  post [2, 1]  in [1, 2]


### (d) Counting trees consistent with pre-order + post-order

In [22]:
def all_trees_pre_post(pre, post):
    n = len(pre)
    if n == 0:
        return [None]
    if n == 1:
        return [BNode(pre[0])]
    size = post.index(pre[1]) + 1       # size of the first subtree
    out = []
    if size == n - 1:                   # one child: side cannot be known
        for sub in all_trees_pre_post(pre[1:], post[:-1]):
            out += [BNode(pre[0], left=sub), BNode(pre[0], right=sub)]
    else:
        for a in all_trees_pre_post(pre[1:1 + size], post[:size]):
            for b in all_trees_pre_post(pre[1 + size:], post[size:-1]):
                out.append(BNode(pre[0], a, b))
    return out


def one_child_nodes(t):
    if t is None:
        return 0
    return ((t.left is None) != (t.right is None)) + one_child_nodes(t.left) + one_child_nodes(t.right)


trees = all_trees_pre_post([1, 2, 3], [3, 2, 1])
print("pre [1,2,3], post [3,2,1]:", len(trees), "trees:", [show_b(t) for t in trees])

trees = all_trees_pre_post(pre, post)
print(f"given 11-node tree: k = {one_child_nodes(t1)}, trees = {len(trees)} (2^k = {2**one_child_nodes(t1)})")
for t in trees:
    print("   ", show_b(t))

pre [1,2,3], post [3,2,1]: 4 trees: ['1(2(3,-),-)', '1(-,2(3,-))', '1(2(-,3),-)', '1(-,2(-,3))']
given 11-node tree: k = 2, trees = 4 (2^k = 4)
    A(B(D,E(H,I(K,-))),C(F,G(J,-)))
    A(B(D,E(H,I(K,-))),C(F,G(-,J)))
    A(B(D,E(H,I(-,K))),C(F,G(J,-)))
    A(B(D,E(H,I(-,K))),C(F,G(-,J)))


### (e) Cost of scanning vs dictionary

In [23]:
def left_chain(n):
    t = None
    for v in range(n):
        t = BNode(v, left=t)
    return t


def right_chain(n):
    t = None
    for v in reversed(range(n)):
        t = BNode(v, right=t)
    return t


def balanced(lo, hi):
    if lo > hi:
        return None
    m = (lo + hi) // 2
    return BNode(m, balanced(lo, m - 1), balanced(m + 1, hi))


print(f"{'n':>5} {'left chain':>10} {'n(n+1)/2':>9} {'right chain':>11} {'balanced':>8} {'dict (ms)':>9}")
for n in [250, 500, 1000, 2000, 4000]:
    comps = [build_scan(preorder(t), inorder(t))[1]
             for t in (left_chain(n), right_chain(n), balanced(0, n - 1))]
    p, i = preorder(left_chain(n)), inorder(left_chain(n))
    t0 = time.perf_counter()
    build_dict(p, i)
    ms = (time.perf_counter() - t0) * 1e3
    print(f"{n:>5} {comps[0]:>10} {n*(n+1)//2:>9} {comps[1]:>11} {comps[2]:>8} {ms:>9.2f}")

    n left chain  n(n+1)/2 right chain balanced dict (ms)
  250      31375     31375         250      989      0.11
  500     125250    125250         500     2222      0.27
 1000     500500    500500        1000     4938      0.54


 2000    2001000   2001000        2000    10870      1.14


 4000    8002000   8002000        4000    23734      4.42


### (f) Repeated values

In [24]:
def all_trees_pre_in(pre, ino):
    n = len(pre)
    if n == 0:
        return [None]
    out = []
    for k in range(n):
        if ino[k] == pre[0] and Counter(pre[1:k + 1]) == Counter(ino[:k]):
            for a in all_trees_pre_in(pre[1:k + 1], ino[:k]):
                for b in all_trees_pre_in(pre[k + 1:], ino[k + 1:]):
                    out.append(BNode(pre[0], a, b))
    return out


def serialise(t):
    # pre-order with (value, has_left, has_right)
    if t is None:
        return []
    return ([(t.val, int(t.left is not None), int(t.right is not None))]
            + serialise(t.left) + serialise(t.right))


def deserialise(data):
    it = iter(data)

    def rec():
        val, l, r = next(it)
        node = BNode(val)
        if l:
            node.left = rec()
        if r:
            node.right = rec()
        return node

    return rec() if data else None


a = BNode(4, BNode(4), BNode(9))
b = BNode(4, None, BNode(4, None, BNode(9)))
for t in (a, b):
    print(f"{show_b(t):<12} pre {preorder(t)}  in {inorder(t)}  serialised {serialise(t)}")
print("trees matching pre [4,4,9], in [4,4,9]:", [show_b(t) for t in all_trees_pre_in([4, 4, 9], [4, 4, 9])])
print("round trip:", all(show_b(deserialise(serialise(t))) == show_b(t) for t in (a, b)))

4(4,9)       pre [4, 4, 9]  in [4, 4, 9]  serialised [(4, 1, 1), (4, 0, 0), (9, 0, 0)]
4(-,4(-,9))  pre [4, 4, 9]  in [4, 4, 9]  serialised [(4, 0, 1), (4, 0, 1), (9, 0, 0)]
trees matching pre [4,4,9], in [4,4,9]: ['4(-,4(-,9))', '4(4,9)']
round trip: True
